In [0]:
bronze_df = spark.table("workspace.default.bronze_dataco")

In [0]:
from pyspark.sql import functions as F

null_counts = bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in bronze_df.columns
])

display(null_counts)

In [0]:
from pyspark.sql import functions as F

silver_df = (
    bronze_df
    #Clean nulls
    .fillna({"customer_lname": "Unknown", "customer_zipcode": 0})
    #Drop column with more nulls, not useful for the analysis
    .drop("order_zipcode")
    #Add derived columns for KPIs
    .withColumn("is_late", F.when(F.col("delivery_status") == "Late delivery", 1).otherwise(0))
    .withColumn("shipping_delay_days", F.col("days_for_shipping_real") - F.col("days_for_shipment_scheduled"))
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("order_yearmonth", F.date_format("order_date", "yyyy-MM"))
)

print(f"Silver layer: {silver_df.count()} rows x {len(silver_df.columns)} columns")

In [0]:
silver_df.write.mode("overwrite").saveAsTable("workspace.default.silver_dataco")